# Fairness Audit: Does a "Group-Blind" Loan Model Treat Both Groups Equally?

## Quick logic summary
- **Goal:** predict `repaid` (1 = good loan) from 5 financial features, *without* the `group` column, then audit the model by group (A vs B).
- **Metrics per group:** *precision* (of approved loans, how many are really repaid), *recall* (of truly good applicants, how many are approved), *approval rate* (share of applicants approved).
- **Key idea:** removing a column does not remove its information. If other columns are correlated with `group`, they act as **proxies**.
- **Fix tried:** group-specific approval thresholds, tuned on a validation set and judged on a separate test set.
- **Result (from the run below):** baseline approves **9.5% of A vs 2.7% of B**. Equalising approval rates closes that gap (0.068 → 0.005) but flips the recall gap (0.131 → -0.061) and leaves the precision gap (0.078).
- **Caveat:** the test set has only ~330 group-B applicants, so numbers can shift by a few points with a different random split.

---

## PART 1: The whole task as a story (plain English)

Imagine a bank with two kinds of applicants, **Group A** and **Group B**. The bank's manager says: *"To be fair, let's hide the group label from the computer. It can't discriminate if it can't see the group."*

So we train a model that only sees income, years employed, credit score, debt ratio and loan amount. It learns to spot who will repay a loan.

Then we play auditor. We bring back the hidden label **only to check results**, and we see the model is **not** equal. Group A gets approved far more often (9.5% vs 2.7%), and among truly good applicants, A's are found 21.6% of the time against only 8.5% for B.

**Why?** The model never saw the label, but it doesn't need to. In this data Group B applicants have lower income (about 44k vs 62k), lower credit scores (640 vs 687) and shorter employment (4.8 vs 7.0 years). Those "neutral" columns are a fingerprint of the group. We prove it by training a second model to guess the group from those columns alone, and it succeeds (AUC 0.82, where 0.5 would be a coin flip). On top of that, in this dataset B's real repayment rate is lower (17.7% vs 28.9%), so the model honestly (but unequally) gives B lower scores.

**The repair:** instead of one cut-off score for everyone, give each group its own cut-off, making it a bit easier for B to pass. We tune these cut-offs on one slice of data and check on a fresh slice. We try two goals:
1. **Equal recall** (equal chance for genuinely good applicants), which is called *Equal Opportunity*.
2. **Equal approval rate** (same share approved in each group), which is called *Demographic Parity*.

**The catch:** you can't have everything at once when the groups' real repayment rates differ. When we force equal approval rates, the recall gap flips direction (B now gets *more* of its good applicants approved than A) and the precision gap does not go away. When we force equal recall, a smaller approval gap remains. That is the fairness trade-off: **pick the fairness definition that matches your values, and accept the cost on the others.**

---

## PART 2: Line-by-line explanation of the code

**Step 1, Imports**
- `pandas`/`numpy`: tables and maths.
- `train_test_split`: divides data; `cross_val_score`: repeated train/test scoring.
- `GradientBoostingClassifier`: our main model (handles non-linear patterns). `LogisticRegression`: a simple model whose weights are easy to read.
- `StandardScaler`: puts features on the same scale so the weights are comparable.
- `precision_score`, `recall_score`, `roc_auc_score`: evaluation metrics.

**Step 2, Load data**
- `pd.read_csv(...)`: reads the file (keep the CSV next to the notebook).
- `FEATURES = [...]`: the 5 allowed inputs. `group` and `applicant_id` are left out, so the model is group-blind.
- `X = df[FEATURES]`: the input table.
- `y = df["repaid"].values`: the answer we predict (1 = repaid).
- `g = df["group"].values`: group labels stored separately, used only for auditing, never for training.

**Step 3, Split**
- `strat = ...`: a label combining `repaid` + `group`, so each split keeps the same mix of both.
- First `train_test_split(test_size=0.4)`: 60% train, 40% held back.
- Second split cuts the held-back 40% in half: 20% **validation** (to tune thresholds) and 20% **test** (to report results honestly).

**Step 4, Train**
- `GradientBoostingClassifier(...).fit(X_tr, y_tr)`: learns from the training rows only.
- `predict_proba(...)[:, 1]`: gives each applicant a repay-probability (score between 0 and 1) for validation (`s_val`) and test (`s_te`).
- `roc_auc_score`: one number for how well the model ranks good vs bad applicants (0.5 = random).

**Step 5, `audit()` function**
- Loops over groups A and B; `m = grp == k` selects that group's rows.
- `pred = (score[m] >= thr[k])`: approve (1) if score passes **that group's** threshold.
- Stores precision, recall and approval rate (`pred.mean()` = share approved).
- `t.loc["gap (A-B)"]`: adds a row with A minus B. A gap of 0 means equal.

**Step 6, Baseline**
- Calls `audit` with threshold 0.5 for both groups and saves it in the dictionary `res`. This is the "before" picture.

**Step 7, Proxy analysis (how the group leaks in)**
- `is_B`: 1 if group B else 0.
- `cross_val_score(... is_B ...)`: trains a model to predict the group from the features. AUC far above 0.5 proves the features reveal the group.
- `LogisticRegression(...).coef_`: how strongly each feature points towards B. Negative = lower values signal B.
- `proxy` table: mean of each feature per group, correlation with being in B, and the weight. Income, credit score and years employed are the main proxies.
- Last `print`: the real repayment rate per group, which differs (a second reason for unequal approvals).

**Step 8, Threshold tuning**
- `thr_for_recall(target)`: for each group, looks at the scores of its truly good applicants on the validation set and picks the cut-off that lets exactly `target` of them through.
- `thr_for_approval(target)`: for each group, picks the cut-off that approves exactly `target` of all its applicants.
- `overall_recall` / `overall_approval`: the targets, taken from the overall behaviour at 0.5 so the bank's total strictness stays about the same.
- `eo_thr`, `dp_thr`: the resulting group-specific thresholds for Fix 1 (Equal Opportunity) and Fix 2 (Demographic Parity).

**Step 9, Re-audit on the test set**
- Runs `audit` with each set of thresholds on data that was never used for tuning, saves the tables, and prints them.

**Step 10, Scoreboard**
- Collects only the `gap (A-B)` row of each scenario into one table, so you can see at a glance which gap shrank and which grew or flipped.

### What to conclude from the scoreboard
| Scenario | Fixed | Effect on the others |
|---|---|---|
| Fix 2: equal approval rate | **Approval rate** gap 0.068 → 0.005 (Demographic Parity) | **Recall** gap flips to -0.061 (B now favoured, Equal Opportunity broken the other way); **precision** gap stays at 0.078 |
| Fix 1: equal recall | **Recall** gap 0.131 → -0.019 (Equal Opportunity) | **Approval rate** gap remains at 0.022; precision gap 0.048 |

*This is not legal advice; which fairness definition is appropriate for real lending depends on regulation and policy.*


In [1]:
# ---------- 1. IMPORTS ----------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# ---------- 2. LOAD DATA, DROP 'group' FROM THE FEATURES ----------
df = pd.read_csv("loan_applications.csv")
FEATURES = ["income", "years_employed", "credit_score", "debt_ratio", "loan_amount"]
X = df[FEATURES]              # model inputs (no group, no id)
y = df["repaid"].values       # 1 = repaid (good loan), 0 = defaulted
g = df["group"].values        # kept aside ONLY for auditing

# ---------- 3. SPLIT: 60% train / 20% validation / 20% test ----------
strat = df["repaid"].astype(str) + df["group"]
X_tr, X_tmp, y_tr, y_tmp, g_tr, g_tmp = train_test_split(X, y, g, test_size=0.4, random_state=42, stratify=strat)
X_val, X_te, y_val, y_te, g_val, g_te = train_test_split(X_tmp, y_tmp, g_tmp, test_size=0.5, random_state=42, stratify=pd.Series(y_tmp).astype(str) + g_tmp)

# ---------- 4. TRAIN THE "GROUP-BLIND" MODEL ----------
model = GradientBoostingClassifier(random_state=42).fit(X_tr, y_tr)
s_val = model.predict_proba(X_val)[:, 1]     # repay-probability on validation
s_te = model.predict_proba(X_te)[:, 1]       # repay-probability on test
print("Model AUC on test:", round(roc_auc_score(y_te, s_te), 3))

# ---------- 5. AUDIT FUNCTION: precision, recall, approval rate per group ----------
def audit(y_true, score, grp, thr):
    rows = {}
    for k in ["A", "B"]:
        m = grp == k                                   # rows belonging to group k
        pred = (score[m] >= thr[k]).astype(int)        # 1 = approve
        rows[k] = {"precision": precision_score(y_true[m], pred, zero_division=0),
                   "recall": recall_score(y_true[m], pred),
                   "approval_rate": pred.mean()}
    t = pd.DataFrame(rows).T
    t.loc["gap (A-B)"] = t.loc["A"] - t.loc["B"]
    return t.round(3)

# ---------- 6. BASELINE: one threshold (0.5) for everybody ----------
res = {"Baseline": audit(y_te, s_te, g_te, {"A": 0.5, "B": 0.5})}
print("\n=== BASELINE (threshold 0.5 for both groups) ===")
print(res["Baseline"])

# ---------- 7. HOW IS THE MODEL "SEEING" THE GROUP? (proxy analysis) ----------
is_B = (g == "B").astype(int)
proxy_auc = cross_val_score(GradientBoostingClassifier(random_state=0), X, is_B, cv=5, scoring="roc_auc").mean()
print("\nAUC of predicting group from the 'neutral' features alone:", round(proxy_auc, 3), "(0.5 = impossible, 1.0 = perfect)")
weights = LogisticRegression(max_iter=2000).fit(StandardScaler().fit_transform(X), is_B).coef_[0]
proxy = pd.DataFrame({"mean_A": df[g == "A"][FEATURES].mean(),
                      "mean_B": df[g == "B"][FEATURES].mean(),
                      "corr_with_B": df[FEATURES].corrwith(pd.Series(is_B)),
                      "proxy_weight": weights}).round(3)
print(proxy.sort_values("proxy_weight"))
print("Real repay rate -> A: %.3f | B: %.3f" % (y[g == "A"].mean(), y[g == "B"].mean()))

# ---------- 8. THRESHOLD TUNING (done on VALIDATION data only) ----------
def thr_for_recall(target):      # cut-off so each group recovers `target` of its true repayers
    return {k: float(np.quantile(s_val[(g_val == k) & (y_val == 1)], 1 - target)) for k in "AB"}
def thr_for_approval(target):    # cut-off so each group approves `target` of its applicants
    return {k: float(np.quantile(s_val[g_val == k], 1 - target)) for k in "AB"}

overall_recall = recall_score(y_val, (s_val >= 0.5).astype(int))
overall_approval = (s_val >= 0.5).mean()
eo_thr = thr_for_recall(overall_recall)          # Fix 1: Equal Opportunity
dp_thr = thr_for_approval(overall_approval)      # Fix 2: Demographic Parity

# ---------- 9. RE-AUDIT ON THE UNTOUCHED TEST SET ----------
res["Fix 1: equal recall"] = audit(y_te, s_te, g_te, eo_thr)
res["Fix 2: equal approval rate"] = audit(y_te, s_te, g_te, dp_thr)
print("\n=== FIX 1: equal recall | thresholds:", {k: round(v, 3) for k, v in eo_thr.items()})
print(res["Fix 1: equal recall"])
print("\n=== FIX 2: equal approval rate | thresholds:", {k: round(v, 3) for k, v in dp_thr.items()})
print(res["Fix 2: equal approval rate"])

# ---------- 10. SCOREBOARD: the A-B gap for every scenario (closer to 0 = fairer) ----------
scoreboard = pd.DataFrame({name: t.loc["gap (A-B)"] for name, t in res.items()}).T
print("\n=== SCOREBOARD: gap (A - B) ===")
print(scoreboard)

Model AUC on test: 0.686

=== BASELINE (threshold 0.5 for both groups) ===
           precision  recall  approval_rate
A              0.659   0.216          0.095
B              0.556   0.085          0.027
gap (A-B)      0.103   0.131          0.068

AUC of predicting group from the 'neutral' features alone: 0.817 (0.5 = impossible, 1.0 = perfect)
                   mean_A     mean_B  corr_with_B  proxy_weight
income          61759.381  44055.484       -0.423        -1.233
credit_score      686.737    639.763       -0.284        -0.695
years_employed      7.021      4.750       -0.157        -0.415
debt_ratio          0.228      0.221       -0.024        -0.063
loan_amount     61617.613  44720.024       -0.231         0.077
Real repay rate -> A: 0.289 | B: 0.177

=== FIX 1: equal recall | thresholds: {'A': 0.518, 'B': 0.406}
           precision  recall  approval_rate
A              0.648   0.184          0.082
B              0.600   0.203          0.060
gap (A-B)      0.048  -0.019  